# Video Inference: HR + SBP + DBP from a Single Video

**Pipeline:** Video → MediaPipe face/hand detection → CAN_2d (PPG) → PPG signals → HR (peak detection) + SBP/DBP (Pulse Transit Time + Windkessel)

---

In [ ]:
# ── Cell 1: Imports & path setup ──
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import cv2, os, sys, shutil, gc, math
from pathlib import Path
from scipy import signal
from scipy.signal import butter, lfilter, lfilter_zi, find_peaks

ROOT = Path(os.getcwd()).resolve()
VBPE = ROOT / '..' / 'rppg_inference_vbpe' if ROOT.name == 'notebooks' else ROOT / 'rppg_inference_vbpe'
sys.path.insert(0, str(VBPE))

os.chdir(str(VBPE))
from face_hands import face_capture, hand_capture
from convert_video import convert_format
from can2dshare import CAN_2d, deep_phys
from vessel_length import get_vessel_length

print('All modules loaded.')
print(f'Workspace: {ROOT}')
print(f'VBPE: {VBPE}')

In [ ]:
# ── Cell 2: Configure inputs ──
VIDEO_PATH = str(VBPE / 'Input_Videos' / 'SL377.mp4')
PHOTO_PATH = str(VBPE / 'Input_Photos' / 'SL377.jpg')
HEIGHT, AGE, BMI = 178, 37, 34.7  # subject demographics

assert os.path.exists(VIDEO_PATH), f'Video not found: {VIDEO_PATH}'
print(f'Video: {VIDEO_PATH}')
print(f'Photo: {PHOTO_PATH}')
print(f'Subject: {HEIGHT}cm, {AGE}yr, BMI={BMI}')

In [ ]:
# ── Cell 3: Face & hand detection → cropped videos ──
print('1/5 Detecting face...')
face_capture(VIDEO_PATH)
print('2/5 Detecting hand...')
hand_capture(VIDEO_PATH)
print('3/5 Converting to MP4...')
for part in ['Face', 'Hand']:
    convert_format(VBPE / 'Results' / 'MP_Videos' / part, VBPE / 'Results' / 'MP_Videos_MP4' / part)
print('  Done.')

In [ ]:
# ── Cell 4: CAN_2d → PPG signals ──
ppg_face_dir = VBPE / 'Results' / 'PPG' / 'Face'
ppg_hand_dir = VBPE / 'Results' / 'PPG' / 'Hand'
os.makedirs(ppg_face_dir, exist_ok=True)
os.makedirs(ppg_hand_dir, exist_ok=True)

print('4/5 Extracting PPG from face...')
deep_phys(str(VBPE / 'Results' / 'MP_Videos_MP4' / 'Face'), str(ppg_face_dir) + '/')
print('5/5 Extracting PPG from hand...')
deep_phys(str(VBPE / 'Results' / 'MP_Videos_MP4' / 'Hand'), str(ppg_hand_dir) + '/')
print('  Done.')

In [ ]:
# ── Cell 5: Compute HR + SBP + DBP ──
def butter_bp(data, low=0.25, high=15, fs=60, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    zi = lfilter_zi(b, a)
    y, _ = lfilter(b, a, data, zi=zi*data[0])
    return y

def peak_detector(s1, s2):
    p1 = signal.argrelextrema(s1, np.less)[0]
    p2 = signal.argrelextrema(s2, np.less)[0]
    min_len = min(len(p1), len(p2)) - 1
    return np.mean(abs(p1[:min_len] - p2[:min_len])) if min_len > 0 else 0

def v_radius(h, a, b): return -0.258 + 0.029*h + 0.006*a + 0.036*b
def v_thick(h, a, b):  return 0.25 + 0.005*a + 0.005*b

# Load PPG CSVs
f_csvs = sorted([f for f in os.listdir(ppg_face_dir) if f.endswith('.csv')])
h_csvs = sorted([f for f in os.listdir(ppg_hand_dir) if f.endswith('.csv')])
assert f_csvs and h_csvs, 'No PPG CSVs found'

face_ppg = pd.read_csv(ppg_face_dir / f_csvs[0]).squeeze().values
hand_ppg = pd.read_csv(ppg_hand_dir / h_csvs[0]).squeeze().values

# Filter
f_filt = butter_bp(face_ppg)
h_filt = butter_bp(hand_ppg)

# ── HR from face PPG (peak detection) ──
peaks, _ = find_peaks(f_filt, distance=20, prominence=0.1)
if len(peaks) > 1:
    rr_intervals = np.diff(peaks) / 60  # seconds (at 60 fps)
    hr_from_ppg = 60.0 / np.mean(rr_intervals)
else:
    hr_from_ppg = 0

# ── BP from PTT ──
lag = peak_detector(f_filt, h_filt)

vessel_len = 0.01 * get_vessel_length(PHOTO_PATH, HEIGHT) if os.path.exists(PHOTO_PATH) else 0.01 * 0.2

lagf = max(abs(lag) / 60, 0.001)
alpha, rho, E0 = 0.017, 1060, 1005
r = 0.001 * 0.5 * v_radius(HEIGHT, AGE, BMI)
h = 0.001 * v_thick(HEIGHT, AGE, BMI)

log_term = (-2/alpha) * np.log(lagf) + (1/alpha) * np.log((2*r*rho*vessel_len**2) / (h*E0))
sbp_val = -0.205 * log_term
dbp_val = -0.12 * log_term

print('═' * 55)
print('           VIDEO INFERENCE RESULTS')
print('═' * 55)
print(f'  Video                     : {Path(VIDEO_PATH).name}')
print(f'  Heart Rate (from PPG)    : {hr_from_ppg:.1f} BPM')
print(f'  Peaks detected           : {len(peaks)}')
print(f'  Pulse Transit Time       : {lag:.1f} frames ({lag/60:.3f}s)')
print(f'  Systolic BP (SBP)        : {sbp_val:.1f} mmHg')
print(f'  Diastolic BP (DBP)       : {dbp_val:.1f} mmHg')
print('═' * 55)

In [ ]:
# ── Cell 6: Visualize PPG signals ──
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
n = 600

axes[0].plot(f_filt[:n], 'steelblue', lw=0.8)
axes[0].set_title('Face PPG (filtered)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(h_filt[:n], 'coral', lw=0.8)
axes[1].set_title('Hand PPG (filtered)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

m = min(300, len(f_filt), len(h_filt))
fn = (f_filt[:m] - f_filt[:m].mean()) / f_filt[:m].std()
hn = (h_filt[:m] - h_filt[:m].mean()) / h_filt[:m].std()
axes[2].plot(fn, 'steelblue', lw=1, alpha=0.8, label='Face PPG')
axes[2].plot(hn, 'coral',   lw=1, alpha=0.8, label='Hand PPG')
axes[2].set_title('PPG Overlay (PTT visible as time shift)')
axes[2].set_xlabel('Frame (60 fps)')
axes[2].set_ylabel('Normalized')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = ROOT / 'outputs' / 'figures' / 'video_inference_ppg.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# ── Cell 7: Summary card ──
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis('off')

info = [
    f'Video: {Path(VIDEO_PATH).name}',
    '',
    f'Heart Rate:  {hr_from_ppg:.1f} BPM',
    f'Systolic BP: {sbp_val:.1f} mmHg',
    f'Diastolic BP:{dbp_val:.1f} mmHg',
    '',
    f'Subject: {HEIGHT}cm, {AGE}yr, BMI={BMI}',
    f'PTT: {lag:.0f} frames'
]
for i, line in enumerate(info):
    w = 'bold' if 'BP:' in line or 'Rate:' in line else 'normal'
    ax.text(0.12, 0.88 - i*0.1, line, fontsize=13, fontweight=w, transform=ax.transAxes, va='top')
ax.set_title('Video Inference — HR + BP', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()

card_path = ROOT / 'outputs' / 'figures' / 'video_inference_card.png'
plt.savefig(card_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {card_path}')

In [ ]:
# ── Cell 8: Save results CSV ──
results = pd.DataFrame([{
    'video': Path(VIDEO_PATH).name,
    'hr_bpm': round(hr_from_ppg, 1),
    'sbp_mmhg': round(float(sbp_val), 1),
    'dbp_mmhg': round(float(dbp_val), 1),
    'ptt_frames': round(lag, 1),
    'peaks_detected': len(peaks),
    'height_cm': HEIGHT,
    'age': AGE,
    'bmi': BMI
}])

csv_path = ROOT / 'outputs' / 'video_inference_results.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(csv_path, index=False)
print(f'Results saved to: {csv_path}')
results

---
## Outputs

| File | Description |
|---|---|
| `outputs/video_inference_results.csv` | HR + SBP + DBP results |
| `outputs/figures/video_inference_ppg.png` | Face & hand PPG signals |
| `outputs/figures/video_inference_card.png` | Summary card |

### How it works:
1. **Face/Hand cropping**: MediaPipe detects face & hand → crops ROIs from each frame
2. **CAN_2d model**: PyTorch model extracts PPG signal from each cropped ROI video
3. **HR**: Peak detection on face PPG → RR interval → BPM
4. **BP**: Cross-correlation between face & hand PPG → PTT → Windkessel physics formula → SBP/DBP

### To run on a different video:
Edit **Cell 2** — set `VIDEO_PATH` to your video file path. Optionally set `PHOTO_PATH` (full-body photo for vessel length) and demographics.